# call run function example
    run("pwd")
    out, _, _ = run("ls", cwd="/home/user/projects", return_output=True)
    out, err, code = run("ls", return_output=True)
    print("Output:", out)
    print("Error:", err)
    print("Exit code:", code)

In [ ]:
# @title Connect google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# %%writefile /content/$WORKING_DIR/app.py

#!/usr/bin/env python3

import os
import re
import shutil
import subprocess
from datetime import datetime
from pathlib import Path

# ==============================
# CONFIG
# ==============================
TODAY = datetime.now().strftime("%d%b%Y").upper()

BASE_DIR = Path(f"/content/{TODAY}")
LLAMA_CPP_DIR = Path(f"{BASE_DIR}/llama.cpp")
HF_CACHE = BASE_DIR / "hf-cache"
OLLAMA_MODELS = Path(f"{BASE_DIR}/ollama_models")

MODEL_NAME = "ggml-org/gemma-4-E2B-it-GGUF"
PORT = 8081

# ==============================
# UTILS
# ==============================
def run(cmd, cwd=None, return_output=False):
    location = f" (in {cwd})" if cwd else ""
    print(f"\n🚀 Running: {cmd}{location}\n")

    result = subprocess.run(
        cmd,
        shell=True,
        cwd=cwd,
        capture_output=True,  # capture stdout + stderr
        text=True             # return as string instead of bytes
    )

    # Always print output
    if result.stdout:
        print("📤 STDOUT:\n", result.stdout)
    if result.stderr:
        print("⚠️ STDERR:\n", result.stderr)

    # Optionally return output
    if return_output:
        return result.stdout, result.stderr, result.returncode

def run_with_logs(cmd, cwd=None):
    location = f" (in {cwd})" if cwd else ""
    print(f"\n🚀 Running: {cmd}{location}\n")

    process = subprocess.Popen(
        cmd,
        shell=True,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # merge stderr into stdout
        text=True,
        bufsize=1
    )

    # Stream output line by line
    for line in process.stdout:
        print(line, end="")

    process.wait()
    return process.returncode

def ensure_dir(path):
    path.mkdir(parents=True, exist_ok=True)

# ==============================
# STEP 1: ENV SETUP
# ==============================
def setup_env():
    os.environ["TODAY_WORK_IN"] = TODAY
    os.environ["HF_HOME"] = str(HF_CACHE)

    print("✅ ENV SET")
    print("TODAY_WORK_IN:", TODAY)
    print("HF_HOME:", HF_CACHE)

# ==============================
# STEP 2: SYSTEM SETUP
# ==============================
def install_system():
    run("apt update -y")
    run("apt install -y build-essential git wget")
    run("pip install -q cmake")

# ------------------------------
# GPU CHECK
# ------------------------------
def has_gpu():
    _, _, code = run("nvidia-smi", return_output=True)
    return code == 0

# ------------------------------
# BUILD
# ------------------------------
def build_llama():
    if not LLAMA_CPP_DIR.exists():
        run("git clone https://github.com/ggerganov/llama.cpp", cwd=BASE_DIR)

    BUILD_NAME="buildCpu"
    cmd_make = f"""
    cmake -B {BUILD_NAME}
    """
    if has_gpu():
        print("🔥 GPU detected → Building with CUDA")
        BUILD_NAME="buildGpu"
        cmd_make = f"""
        cmake -B {BUILD_NAME} \
          -DGGML_CUDA=ON \
          -DCMAKE_CUDA_ARCHITECTURES=native
        """

    else:
        print("💻 No GPU detected → Building CPU version")


    if (LLAMA_CPP_DIR / BUILD_NAME).exists():
        print(f"✅ BUILD EXIST {BUILD_NAME}")
    else:
      run(cmd_make, cwd=LLAMA_CPP_DIR)
      run(f"cmake --build {BUILD_NAME} -j$(nproc)", cwd=LLAMA_CPP_DIR)
      run(f"./{BUILD_NAME}/bin/llama-cli -h", cwd=LLAMA_CPP_DIR)

# ==============================
# STEP 4: HF CACHE
# ==============================
def copy_hf_cache():
    BUILD_NAME="buildCpu"

    if has_gpu():
      BUILD_NAME="buildGpu"

    run(f"./{BUILD_NAME}/bin/llama-cli -hf {MODEL_NAME}",cwd=LLAMA_CPP_DIR)
    print("✅ Copied HF cache")

# ==============================
# STEP 5: START LLAMA SERVER
# ==============================
def start_llama_server():
    BUILD_NAME="buildCpu"

    if has_gpu():
      BUILD_NAME="buildGpu"

    cmd = f"""
    ./{BUILD_NAME}/bin/llama-server \
      -hf {MODEL_NAME} \
      -ngl 99 \
      --flash-attn on \
      --host 0.0.0.0 \
      --port {PORT} \
      > llama.log 2>&1 &
    """
    run(cmd,cwd=LLAMA_CPP_DIR)
    print("✅ Started LLAMA server")

def check_server():
    # out, err, code = run(f"curl http://localhost:{PORT}/v1/models", return_output=True)
    # print("Output:", out)
    # print("Error:", err)
    # print("Exit code:", code)
    run_with_logs(f"curl http://localhost:{PORT}/v1/models | jq")

# ==============================
# Download cloudflared (for public URL)
# ==============================
def cloudflared_download():
    run(f"wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", cwd=BASE_DIR)
    run("chmod +x cloudflared-linux-amd64", cwd=BASE_DIR)
    print(f"cloudflared downloaded {BASE_DIR}")

# ==============================
# Run cloudflared (for public URL)
# ==============================
def cloudflared_run():
    run_cloud=f"""
    ./cloudflared-linux-amd64 tunnel \
      --url http://localhost:{PORT} \
      > cloudflared.log 2>&1 &
    """

    run_cloud1=f"""
    ./cloudflared-linux-amd64 tunnel \
      --url http://localhost:{PORT}
    """

    run_with_logs(run_cloud1, cwd=BASE_DIR)
    # run(run_cloud, cwd=BASE_DIR)
    print(f"cloudflared run and check logs")

# ==============================
# LLAMA cpp upload into drive
# ==============================
def llama_drive_upload():
    drive_dir="/content/drive/MyDrive/Colab Notebooks"
    run(f"cp -r {LLAMA_CPP_DIR} '{drive_dir}'")
    print(f"uploaded to drive: {LLAMA_CPP_DIR} {drive_dir}")

# ==============================
# LLAMA cpp download from drive
# ==============================
def llama_drive_download():
    drive_dir="/content/drive/MyDrive/Colab Notebooks/llama.cpp"
    if not LLAMA_CPP_DIR.exists():
        print(f"{LLAMA_CPP_DIR} is copy")
        run(f"cp -r '{drive_dir}' {BASE_DIR} ")
    else:
      print(f"{LLAMA_CPP_DIR} already exist")
    print(f"downloaded from drive: {drive_dir} {BASE_DIR}")

# ==============================
# MAIN PIPELINE
# ==============================
def main():
    run("pwd")
    ensure_dir(BASE_DIR) # create working dir
    # setup_env()
    # install_system()
    # build_llama()
    # copy_hf_cache()
    # start_llama_server()
    # check_server()
    # cloudflared_download()
    # cloudflared_run()

    # llama_drive_upload() # upload to drive
    # llama_drive_download() # download from drive
    print("\n🎉 ALL DONE SUCCESSFULLY!")
# ==============================
if __name__ == "__main__":
    main()


🚀 Running: pwd

📤 STDOUT:
 /content


🚀 Running: nvidia-smi

⚠️ STDERR:
 /bin/sh: 1: nvidia-smi: not found


🚀 Running: 
    ./buildCpu/bin/llama-server       -hf ggml-org/gemma-4-E2B-it-GGUF       -ngl 99       --flash-attn on       --host 0.0.0.0       --port 8081       > llama.log 2>&1 &
     (in /content/09MAY2026/llama.cpp)

✅ Started LLAMA server

🚀 Running: wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 (in /content/09MAY2026)


🚀 Running: chmod +x cloudflared-linux-amd64 (in /content/09MAY2026)

cloudflared downloaded /content/09MAY2026

🚀 Running: 
    ./cloudflared-linux-amd64 tunnel       --url http://localhost:8081
     (in /content/09MAY2026)

2026-05-09T12:43:45Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of U